# Data Preprocessing & Feature Engineering

It is the main nootebook related to Data Preprocessing & Feature Engineering of https://www.kaggle.com/datasets/sahistapatel96/bankadditionalfullcsv dataset.

Previous notbook related to EDA - https://github.com/Maxstef/ml-bank-additional-project/blob/main/notebooks/01_eda.ipynb




## Notebook initialization

In [1]:
# Notebook initialization
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

ROOT = Path.cwd()

while not (ROOT / "src").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

# print("Project root:", ROOT)

## Imports & Load data

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_selector
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

from src.data import load_raw_data

raw_df = load_raw_data()
raw_df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [3]:
target_col = "y"

## Split Data Train & Validation

Further during modeling, cross-validation can be used.

Here, the dataset is split into training and validation sets with an 80/20 ratio (4:1). The split is stratified by the target variable to preserve the class distribution due to target class imbalance.

In [4]:
from src.data import split_train_val

train_df, val_df = split_train_val(raw_df, stratify_col=target_col)
train_df.shape, val_df.shape

((32950, 21), (8238, 21))

In [5]:
train_df[target_col].value_counts(normalize=True), val_df[target_col].value_counts(normalize=True)

(y
 no     0.887344
 yes    0.112656
 Name: proportion, dtype: float64,
 y
 no     0.887351
 yes    0.112649
 Name: proportion, dtype: float64)

## Custom Numeric Column Transformers

The plan is to use scikit-learn Pipelines for modeling.
To keep the logic simple and readable, goal is to implement a few custom transformers to be reused in future pipeline experiments.


### Domain Specific Column Transformers

Column Transformers specific for <a href="https://www.kaggle.com/datasets/sahistapatel96/bankadditionalfullcsv">bank additional</a> dataset

#### CampaignPrevTransformer
Creates `campaign_prev = min(campaign - 1, 5)`

In [6]:
from src.preprocessing import CampaignPrevTransformer
campaign_tr = CampaignPrevTransformer()

campaign_tr_df = campaign_tr.fit_transform(train_df)

campaign_tr_df[campaign_tr_df["campaign"] > 4][["campaign", "campaign_prev"]].head(6)

,campaign,campaign_prev
14697,5,4
17537,5,4
1595,5,4
4216,5,4
6347,9,5
9348,7,5


In [7]:
campaign_tr_df["campaign_prev"].value_counts().sort_index()

campaign_prev
0    14121
1     8469
2     4300
3     2116
4     1255
5     2689
Name: count, dtype: int64

#### PdaysTransformer

Depending on the `mode` parameter, this transformer converts `pdays` into:

 - a **binary feature**: `pdays_contacted = 1` if `pdays != 999`, otherwise `0`
 - **categorical groups**, such as:
   - `not_contacted_before`
   - `contacted_recently` (e.g., `< 7` days)
   - `contacted_long_ago` (e.g., `≥ 7` days)

In [8]:
from src.preprocessing import PdaysTransformer

# binary case
pdays_tr = PdaysTransformer(mode="binary")
pdays_tr_df = pdays_tr.fit_transform(train_df)
pdays_tr_df[["pdays", "pdays_contacted"]].iloc[-10:-5]

,pdays,pdays_contacted
15913,999,0
30274,2,1
24607,999,0
27529,999,0
34253,999,0


In [9]:
# group case
pdays_tr = PdaysTransformer(mode="group", recent_days=7)
pdays_tr_df = pdays_tr.fit_transform(train_df)
pdays_tr_df[pdays_tr_df["pdays"] < 999][["pdays", "pdays_group"]].head()

,pdays,pdays_group
39226,3,contacted_recently
39224,6,contacted_recently
40273,3,contacted_recently
35478,10,contacted_long_ago
39714,8,contacted_long_ago


### Generic Transformers

Custom column transformers that are **not dataset-specific** and can be **reused with any dataset**.

#### ColumnDropper

This step can be used:
 - as the **first step** to **drop features that will not be used in the model** at an early stage
 - to **remove columns that have already been preprocessed** and are **no longer relevant for modeling**

In [10]:
from src.preprocessing import ColumnDropper

cols_to_drop = ["duration", "contact", "housing", "loan"]

pipe_auto = make_pipeline(ColumnDropper(columns=cols_to_drop))

drop_tr_df = pipe_auto.fit_transform(train_df)

print("Original columns:", train_df.shape[1])
print("New columns:", drop_tr_df.shape[1])
print("Dropped columns:", set(train_df.columns) - set(drop_tr_df.columns))

Original columns: 21
New columns: 17
Dropped columns: {'duration', 'housing', 'loan', 'contact'}


#### OutlierCapper

 - Caps values in a specified column.
 - If `cap` is not provided, calculates it using the IQR method (Q3 + 1.5*IQR by default)

In [11]:
from src.preprocessing import OutlierCapper

# Cap campaign at 6 explicitly
capper_tr = OutlierCapper(column="campaign", cap=6)

capper_tr_df = capper_tr.fit_transform(train_df)

# Check results
capper_tr_df["campaign"].value_counts().sort_index()

campaign
1    14121
2     8469
3     4300
4     2116
5     1255
6     2689
Name: count, dtype: int64

In [12]:
# Cap campaign at 6 implicitly using iqr otliers method
capper_tr_iqr = OutlierCapper(column="campaign")
capper_tr_iqr_df = capper_tr_iqr.fit_transform(train_df)
capper_tr_iqr_df["campaign"].value_counts().sort_index()

campaign
1.0    14121
2.0     8469
3.0     4300
4.0     2116
5.0     1255
6.0     2689
Name: count, dtype: int64

#### ColumnArithmetic

- Applies an arithmetic operation (**add, subtract, multiply, divide**) to a column.
- When used together with **OutlierCapper**, it can reproduce the same transformation as the domain-specific `CampaignPrevTransformer`.
- Since feature scaling may be applied at later preprocessing stages, `ColumnArithmetic` may be **redundant for this specific `campaign` case**.

In [13]:
from src.preprocessing import ColumnArithmetic

pipe_ca_oc = make_pipeline(
    ColumnArithmetic(column="campaign", operation="subtract", value=1),
    OutlierCapper(column="campaign", cap=5),
)

arithmetic_tr_df = pipe_ca_oc.fit_transform(train_df)
arithmetic_tr_df["campaign"].value_counts().sort_index()

campaign
0    14121
1     8469
2     4300
3     2116
4     1255
5     2689
Name: count, dtype: int64

#### NumericBinner

Transforms a numeric column into categorical bins based on specified thresholds.

In [14]:
from src.preprocessing import NumericBinner

# example for age column
age_binner_tr = NumericBinner(
    column="age",
    bins=[0, 25, 60, 120],
    labels=["young", "middle", "old"]
)

age_binner_tr_df = age_binner_tr.fit_transform(train_df)

age_binner_tr_df[["age", "age_group"]].head(5)

,age,age_group
25611,49,middle
26010,37,middle
40194,78,old
297,36,middle
36344,59,middle


In [15]:
# can be used for pdays column too, but maybe not the best approach to handle this special 999 value
pdays_binner_tr = NumericBinner(
    column="pdays",
    bins=[0, 7, 998, 1000],
    labels=["contacted_recently", "contacted_long_ago", "not_contacted_before"]
)

pdays_binner_tr_df = pdays_binner_tr.fit_transform(train_df)
pdays_binner_tr_df[pdays_binner_tr_df["pdays"] < 999][["pdays", "pdays_group"]].head()

,pdays,pdays_group
39226,3,contacted_recently
39224,6,contacted_recently
40273,3,contacted_recently
35478,10,contacted_long_ago
39714,8,contacted_long_ago


#### ConditionalMapper

Applies rule-based mapping to create a new column

In [16]:
from src.preprocessing import ConditionalMapper

# better approach for pdays column handling than above NumericBinner
def pdays_not_contacted(x):
    return x == 999

def pdays_contacted_recently(x):
    return x < 7

def pdays_contacted_long_ago(x):
    return (x >= 7) & (x != 999)

pdays_rules = [
    (pdays_not_contacted, "not_contacted_before"),
    (pdays_contacted_recently, "contacted_recently"),
    (pdays_contacted_long_ago, "contacted_long_ago"),
]
mapper_pdays_tr = ConditionalMapper(
    column="pdays",
    new_column="pdays_group",
    rules=pdays_rules,
)

mapper_pdays_tr_df = mapper_pdays_tr.fit_transform(train_df)
mapper_pdays_tr_df[mapper_pdays_tr_df["pdays"] < 999][["pdays", "pdays_group"]].head()

,pdays,pdays_group
39226,3,contacted_recently
39224,6,contacted_recently
40273,3,contacted_recently
35478,10,contacted_long_ago
39714,8,contacted_long_ago


#### MappingTransformer

Map values from an existing column to a new column using a dictionary

In [17]:
from src.preprocessing import MappingTransformer
from src.preprocessing.mappings import month_map

mapper_month_tr = MappingTransformer(
    column="month",
    new_column="month_num",
    mapping=month_map,
)

mapper_month_tr_df = mapper_month_tr.fit_transform(train_df)
mapper_month_tr_df[["month", "month_num"]].head()

,month,month_num
25611,nov,11
26010,nov,11
40194,jul,7
297,may,5
36344,jun,6


#### CyclicalEncoder

Encode cyclical features using **sine and cosine transformations**.

Useful for: month, day_of_week, hour, etc.

**Note:** Sine alone is not sufficient to uniquely represent each value in the cycle because of periodic symmetry. Using **both sine and cosine** ensures that each value has a **unique representation** and preserves the cyclical relationship.

In [18]:
from src.preprocessing.mappings import month_map, dow_map

cyclical_train_df = train_df.copy()

cyclical_train_df["month_num"] = cyclical_train_df["month"].map(month_map)
cyclical_train_df["day_of_week_num"] = cyclical_train_df["day_of_week"].map(dow_map)

In [19]:
from src.preprocessing import CyclicalEncoder

# Month (1–12)
month_encoder = CyclicalEncoder(column="month_num", max_value=12)
cyclical_train_df = month_encoder.fit_transform(cyclical_train_df)

# Day of week (assuming Monday=1, Sunday=7)
dow_encoder = CyclicalEncoder(column="day_of_week_num", max_value=5)
cyclical_train_df = dow_encoder.fit_transform(cyclical_train_df)

# Check
cyclical_train_df.head()[["month", "month_num_sin", "month_num_cos", "day_of_week", "day_of_week_num_sin", "day_of_week_num_cos"]]

,month,month_num_sin,month_num_cos,day_of_week,day_of_week_num_sin,day_of_week_num_cos
25611,nov,-5.000000e-01,0.866025,wed,-0.587785,-0.809017
26010,nov,-5.000000e-01,0.866025,wed,-0.587785,-0.809017
40194,jul,-5.000000e-01,-0.866025,mon,0.951057,0.309017
297,may,5.000000e-01,-0.866025,mon,0.951057,0.309017
36344,jun,1.224647e-16,-1.000000,tue,0.587785,-0.809017


## Custom Categorical Column Transformers

### CategoricalEncoder

Use the custom `CategoricalEncoder` instead of `sklearn.preprocessing.OneHotEncoder` for **flexibility, clarity, and maximum control**.

One-hot encode categorical columns with options to:
- **Drop specified categories**
- **Combine multiple categories into one**


Note: unknown values encoded as all zeros (same as `handle_unknown="ignore"` param for sklearn OneHotEncoder)

In [20]:
from src.preprocessing import CategoricalEncoder

cat_cols = ["job", "marital", "education", "default"]

drop_cats = {
    "job": ["unknown"],
    "marital": ["unknown"],
    "education": []  # We'll combine unknown and illiterate here
}

combine_cats = {
    "education": {"other": ["unknown", "illiterate"]},
    "default": {"risk": ["unknown", "yes"]}
}

encoder_tr = CategoricalEncoder(
    columns=cat_cols,
    drop_categories=drop_cats,
    combine_categories=combine_cats
)

encoder_tr_df = encoder_tr.fit_transform(train_df)
encoder_tr_df.iloc[:, -15:].head()

,job_student,job_technician,job_unemployed,marital_divorced,marital_married,marital_single,education_basic.4y,education_basic.6y,education_basic.9y,education_high.school,education_other,education_professional.course,education_university.degree,default_no,default_risk
25611,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1
26010,0,0,0,0,1,0,0,0,0,0,0,0,1,1,0
40194,0,0,0,0,1,0,1,0,0,0,0,0,0,1,0
297,0,0,0,0,1,0,0,0,0,0,0,0,1,1,0
36344,0,0,0,1,0,0,0,0,0,0,0,0,1,1,0


In [21]:
# encode all (no dropping, no combining)
all_cat_cols = raw_df.select_dtypes(include=object).columns.drop(target_col)

encoder_all_tr = CategoricalEncoder(
    columns=all_cat_cols
)

encoder_all_tr_df = encoder_all_tr.fit_transform(train_df)
encoder_all_tr_df.iloc[:, -30:].head()

/var/folders/0s/q_d47bvj0hq4rs_6q_06d63m0000gn/T/ipykernel_11063/536879140.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  all_cat_cols = raw_df.select_dtypes(include=object).columns.drop(target_col)


,education_unknown,default_no,default_unknown,default_yes,housing_no,housing_unknown,housing_yes,loan_no,loan_unknown,loan_yes,...,month_oct,month_sep,day_of_week_fri,day_of_week_mon,day_of_week_thu,day_of_week_tue,day_of_week_wed,poutcome_failure,poutcome_nonexistent,poutcome_success
25611,0,0,1,0,1,0,0,1,0,0,...,0,0,0,0,0,0,1,0,1,0
26010,0,1,0,0,1,0,0,1,0,0,...,0,0,0,0,0,0,1,1,0,0
40194,0,1,0,0,1,0,0,1,0,0,...,0,0,0,1,0,0,0,0,1,0
297,0,1,0,0,0,0,1,1,0,0,...,0,0,0,1,0,0,0,0,1,0
36344,0,1,0,0,1,0,0,1,0,0,...,0,0,0,0,0,1,0,0,1,0


## Combine all into Pipeline

Combine above custom transformers into sklearn Pipeline and transform raw dataframe with it.

Some notes:
 - As an experiment, use cyclical encoding for month and day_of_week.
 - Keep all socio-economic attributes as numeric in this initial preprocessing pipeline.
 - The target column is included as categorical.
 - No handling of potential missing values (imputation) is performed for future (real-world) test data.


In [22]:
from src.preprocessing import (
    ColumnDropper,
    ColumnArithmetic,
    OutlierCapper,
    NumericBinner,
    PdaysTransformer,
    ConditionalMapper,
    MappingTransformer,
    CategoricalEncoder,
    AutoCategoricalEncoder,
    CyclicalEncoder,
)

from src.preprocessing.mappings import month_map, dow_map

feature_eng_pipeline = Pipeline([
    # Drop irrelevant columns
    ("drop_cols_initial", ColumnDropper(columns=["duration", "contact", "housing", "loan"])),
    
    # Transform campaign -> campaign_prev
    ("campaign_prev", Pipeline([
        ("subtract_1", ColumnArithmetic(column="campaign", operation="subtract", value=1)),
        ("cap_outliers", OutlierCapper(column="campaign", cap=5))
    ])),
    
    # Transform pdays
    ("pdays_transform", PdaysTransformer(mode="group", recent_days=7)),
    
    # Month & day_of_week mapped to numeric
    ("month_mapper", MappingTransformer(column="month", mapping=month_map, new_column="month_num")),
    ("dow_mapper", MappingTransformer(column="day_of_week", mapping=dow_map, new_column="dow_num")),

    # Drop categorical month and day_of_week columns after transformation to numeric
    ("drop_cols_month_day", ColumnDropper(columns=["month", "day_of_week"])),
    
])

# Categorical encoding
drop_cats = {
    "job": ["unknown"],
    "marital": ["unknown"],
    "education": ["unknown", "illiterate"],
    "poutcome": ["nonexistent"]
}
combine_cats = {
    "default": {"risk": ["unknown", "yes"]}
}

# AutoCategoricalEncoder does ot require columns param on creation and select all categorical columns automatically during pipelining
cat_encoder = AutoCategoricalEncoder(
    drop_categories=drop_cats,
    combine_categories=combine_cats
)

# Cyclical encoding for month/day_of_week numeric columns - experimental (alternative to one hot encodign and ordinal encoding)
cyclical_cols = ["month_num", "dow_num"]
cyclical_encoder = Pipeline([
    ("cyclical", CyclicalEncoder(column="month_num", max_value=12)),
    ("cyclical_dow", CyclicalEncoder(column="dow_num", max_value=5)), # there are only weekdays (mon-fri) in dataset

    # Drop numeric month and day_of_week columns after transformation to cyclical
    ("drop_cols_month_day_num", ColumnDropper(columns=["month_num", "dow_num"])),
])

# preprocess numeric with standardize scaling - use sklearn StandardScaler
num_encoder = Pipeline(
    [("scaler", StandardScaler())]
)

final_preprocessor = ColumnTransformer(transformers=[
    ("cat", cat_encoder, make_column_selector(dtype_include="object")),         # all categorical
    ("numeric", num_encoder, make_column_selector(dtype_include="number")),     # all numeric
])

preprocessing_pipeline = Pipeline([
    ("feature_engineering", feature_eng_pipeline),
    ("cyclical", cyclical_encoder),
    ("preprocessing", final_preprocessor)
])

In [23]:
transformed_df = preprocessing_pipeline.fit_transform(raw_df)
print(transformed_df.shape)

(41188, 42)


In [24]:
preprocessing_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('feature_engineering', ...), ('cyclical', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('drop_cols_initial', ...), ('campaign_prev', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,columns,"['duration', 'contact', ...]"
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for

### Features names out

In [25]:
feature_names = preprocessing_pipeline.named_steps['preprocessing'].get_feature_names_out()
print(len(feature_names))
print(feature_names)

42
['cat__job_admin.' 'cat__job_blue-collar' 'cat__job_entrepreneur'
 'cat__job_housemaid' 'cat__job_management' 'cat__job_retired'
 'cat__job_self-employed' 'cat__job_services' 'cat__job_student'
 'cat__job_technician' 'cat__job_unemployed' 'cat__marital_divorced'
 'cat__marital_married' 'cat__marital_single' 'cat__education_basic.4y'
 'cat__education_basic.6y' 'cat__education_basic.9y'
 'cat__education_high.school' 'cat__education_professional.course'
 'cat__education_university.degree' 'cat__default_no' 'cat__default_risk'
 'cat__poutcome_failure' 'cat__poutcome_success' 'cat__y_no' 'cat__y_yes'
 'cat__pdays_group_contacted_long_ago'
 'cat__pdays_group_contacted_recently'
 'cat__pdays_group_not_contacted_before' 'numeric__age'
 'numeric__campaign' 'numeric__pdays' 'numeric__previous'
 'numeric__emp.var.rate' 'numeric__cons.price.idx'
 'numeric__cons.conf.idx' 'numeric__euribor3m' 'numeric__nr.employed'
 'numeric__month_num_sin' 'numeric__month_num_cos' 'numeric__dow_num_sin'
 'numer

### Save preprocessed data

In [26]:
preprocessed_df = pd.DataFrame(
    transformed_df,
    columns=feature_names,
    index=raw_df.index
)

In [27]:
from src.config import DATA_PROCESSED

preprocessed_df.to_csv(
    DATA_PROCESSED / "bank-additional-preprocessed.csv.zip",
    index=False,
    compression="zip"
)

### Save preprocessing pipeline

In [28]:
import joblib
from src.config import MODELS_DIR

joblib.dump(
    preprocessing_pipeline,
    MODELS_DIR / "full_preprocessing_pipeline.joblib"
);

#### Smoke test on validation data 

In [29]:
pipe = joblib.load(MODELS_DIR / "full_preprocessing_pipeline.joblib")
val_df_transformed = pipe.transform(val_df)

In [30]:
val_df

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
14455,32,management,divorced,university.degree,no,no,no,cellular,jul,tue,...,5,999,0,nonexistent,1.4,93.918,-42.7,4.961,5228.1,no
36380,37,unemployed,unknown,university.degree,no,no,no,cellular,jun,tue,...,1,999,0,nonexistent,-2.9,92.963,-40.8,1.262,5076.2,no
40076,73,retired,divorced,professional.course,unknown,yes,no,cellular,jul,thu,...,2,999,0,nonexistent,-1.7,94.215,-40.3,0.810,4991.6,no
10778,44,entrepreneur,married,basic.4y,unknown,no,no,telephone,jun,tue,...,2,999,0,nonexistent,1.4,94.465,-41.8,4.961,5228.1,no
27939,28,admin.,single,high.school,no,no,no,cellular,mar,fri,...,2,999,0,nonexistent,-1.8,92.843,-50.0,1.531,5099.1,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33359,48,management,married,basic.9y,no,yes,no,cellular,may,tue,...,1,999,0,nonexistent,-1.8,92.893,-46.2,1.291,5099.1,yes
34998,30,services,single,high.school,no,yes,no,cellular,may,fri,...,1,999,0,nonexistent,-1.8,92.893,-46.2,1.250,5099.1,no
39861,33,services,married,high.school,no,no,no,cellular,jun,mon,...,1,999,1,failure,-1.7,94.055,-39.8,0.720,4991.6,no
3920,44,blue-collar,married,basic.6y,no,yes,yes,telephone,may,mon,...,7,999,0,nonexistent,1.1,93.994,-36.4,4.858,5191.0,no


In [31]:
# check first row details
(
    val_df_transformed[0][np.argmax(feature_names == "cat__job_management")],                   # job - management
    val_df_transformed[0][np.argmax(feature_names == "cat__marital_divorced")],                 # marital - divorced
    val_df_transformed[0][np.argmax(feature_names == "cat__education_university.degree")],      # education - university.degree	
    val_df_transformed[0][np.argmax(feature_names == "cat__default_no")],                       # default - no
    val_df_transformed[0][np.argmax(feature_names == "cat__default_risk")],
)

(np.float64(1.0),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(0.0))

## Socio-economic context columns options

Preprocessing options for socio-economic context columns - bin based on previous EDA

In [32]:
soceco_pipeline = Pipeline([
    (
        "emp.var.rate_numbin", 
        NumericBinner(
            column="emp.var.rate",
            bins=[-np.inf, -0.5, np.inf],
            labels=["low", "high"]
        )
    ),
    (
        "euribor3m_numbin", 
        NumericBinner(
            column="euribor3m",
            bins=[-np.inf, 3, np.inf],
            labels=["low", "high"]
        )
    ),
    (
        "nr.employed_numbin", 
        NumericBinner(
            column="nr.employed",
            bins=[-np.inf, 5077, 5150, np.inf],
            labels=["low", "medium", "high"]
        )
    ),
    (
        "cons.price.idx_numbin", 
        NumericBinner(
            column="cons.price.idx",
            bins=[-np.inf, 92.75, 93.5, np.inf],
            labels=["low", "medium", "high"]
        )
    ),
    (
        "cons.conf.idx_numbin", 
        NumericBinner(
            column="cons.conf.idx",
            bins=[-np.inf, -45, -40, -35, np.inf],
            labels=["very low", "low", "medium", "high"]
        )
    ),
])

soceco_cols = ["emp.var.rate", "cons.price.idx", "cons.conf.idx", "euribor3m", "nr.employed"]

soceco_transformed_df = soceco_pipeline.fit_transform(raw_df[soceco_cols])

In [33]:
from src.config import DATA_PROCESSED

soceco_transformed_df.to_csv(
    DATA_PROCESSED / "social-economic-context-binned.csv.zip",
    index=False,
    compression="zip"
)

## Age columns options

Different bin options for age to be used during modeling

In [39]:
age_pipeline = Pipeline([
    (
        "age_numbin",
        NumericBinner(
            column="age",
            bins=[0, 25, 58, 120],
            labels=["young", "middle", "old"]
        )
    ),
    (
        "age_numbin_2",
        NumericBinner(
            column="age",
            new_column="age_range",
            bins=[0, 24, 29, 39, 49, 59, 120],
            labels=["≤24", "25-29", "30-39", "40-49", "50-59", "60+"]
        )
    ),
])

age_transformed_df = age_pipeline.fit_transform(raw_df[["age"]])

In [40]:
age_transformed_df.to_csv(
    DATA_PROCESSED / "age-binned.csv.zip",
    index=False,
    compression="zip"
)

## Month & day of week options

Different bin options for month abd day_of_week to be used during modeling:
 - one hot encoding
 - chronological (ordinal) encoding
 - cyclical encoding

In [42]:
calendar_cols = ["month", "day_of_week"]

calendar_pipeline = Pipeline([
    ("month_mapper", MappingTransformer(column="month", mapping=month_map, new_column="month_num")),
    ("dow_mapper", MappingTransformer(column="day_of_week", mapping=dow_map, new_column="dow_num")),
    ("cyclical", CyclicalEncoder(column="month_num", max_value=12)),
    ("cyclical_dow", CyclicalEncoder(column="dow_num", max_value=5)),
    ("calendar_one_hot_encode", CategoricalEncoder(columns=calendar_cols)),
])

calendar_transformed_df = calendar_pipeline.fit_transform(raw_df[calendar_cols])

In [43]:
calendar_transformed_df.to_csv(
    DATA_PROCESSED / "calendar-context-options.csv.zip",
    index=False,
    compression="zip"
)